# Module 0 — Setup

**Time**: ~10 minutes (run once, instructor-only on Vocareum)

This notebook lands the public datasets we use across the workshop:
- ~20 CISA advisory PDFs (for `ai_parse_document` in Module 1)
- CISA KEV catalog snapshot
- NIST NVD CVE snapshot (recent CVEs only)
- MITRE ATT&CK STIX bundle
- A handful of DoD STIG XCCDF excerpts (Windows, Linux, Cisco IOS)
- Synthetic `affected_assets` table (so attendees can join CVEs to a fake DoD asset inventory)

**You only need to run this once per workshop.** On Vocareum, the lifecycle scripts handle this automatically; locally, run top-to-bottom on a serverless cluster.

## 1. Catalog, schema, volumes

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS disa_workshop;
CREATE SCHEMA IF NOT EXISTS disa_workshop.threat_intel;
CREATE VOLUME IF NOT EXISTS disa_workshop.threat_intel.raw_advisories;
CREATE VOLUME IF NOT EXISTS disa_workshop.threat_intel.raw_stigs;
USE CATALOG disa_workshop;
USE SCHEMA threat_intel;

## 2. CISA KEV catalog

The Known Exploited Vulnerabilities catalog is the de-facto authoritative list of CVEs DISA cares about. We pull the live JSON feed and land it as a Delta table.

In [ ]:
import requests
import json
from pyspark.sql import functions as F

KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
raw = requests.get(KEV_URL, timeout=30).json()
vulns = raw["vulnerabilities"]
print(f"Fetched {len(vulns)} KEV entries (catalog version {raw['catalogVersion']})")

df = spark.createDataFrame(vulns)
df = df.withColumn("dateAdded", F.to_date("dateAdded")).withColumn("dueDate", F.to_date("dueDate"))
df.write.mode("overwrite").saveAsTable("disa_workshop.threat_intel.kev_catalog")
df.limit(5).display()

## 3. NIST NVD — recent CVEs

We scope to CVEs from 2024 onward to keep the dataset small (~30 MB).

In [ ]:
import gzip
from io import BytesIO

NVD_FEEDS = [
    "https://nvd.nist.gov/feeds/json/cve/1.1/nvdcve-1.1-2024.json.gz",
    "https://nvd.nist.gov/feeds/json/cve/1.1/nvdcve-1.1-2025.json.gz",
]

rows = []
for url in NVD_FEEDS:
    resp = requests.get(url, timeout=120)
    data = json.loads(gzip.decompress(resp.content))
    for item in data["CVE_Items"]:
        cve_id = item["cve"]["CVE_data_meta"]["ID"]
        descs = item["cve"]["description"]["description_data"]
        description = descs[0]["value"] if descs else ""
        cvss_v3 = item.get("impact", {}).get("baseMetricV3", {}).get("cvssV3", {})
        rows.append({
            "cve_id": cve_id,
            "description": description,
            "published_date": item["publishedDate"][:10],
            "cvss_score": cvss_v3.get("baseScore"),
            "cvss_severity": cvss_v3.get("baseSeverity"),
            "attack_vector": cvss_v3.get("attackVector"),
        })

print(f"Parsed {len(rows)} CVEs")
df = spark.createDataFrame(rows).withColumn("published_date", F.to_date("published_date"))
df.write.mode("overwrite").saveAsTable("disa_workshop.threat_intel.cves")

## 4. MITRE ATT&CK

STIX 2.1 bundle. We flatten to two tables: `attack_techniques` and `attack_groups`.

In [ ]:
ATTACK_URL = "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json"
bundle = requests.get(ATTACK_URL, timeout=120).json()

techniques, groups = [], []
for obj in bundle["objects"]:
    if obj["type"] == "attack-pattern":
        ext_refs = obj.get("external_references", [])
        attack_id = next((r["external_id"] for r in ext_refs if r.get("source_name") == "mitre-attack"), None)
        techniques.append({
            "technique_id": attack_id,
            "name": obj.get("name"),
            "description": obj.get("description", "")[:2000],
            "is_subtechnique": obj.get("x_mitre_is_subtechnique", False),
            "platforms": obj.get("x_mitre_platforms", []),
        })
    elif obj["type"] == "intrusion-set":
        ext_refs = obj.get("external_references", [])
        group_id = next((r["external_id"] for r in ext_refs if r.get("source_name") == "mitre-attack"), None)
        groups.append({
            "group_id": group_id,
            "name": obj.get("name"),
            "aliases": obj.get("aliases", []),
            "description": obj.get("description", "")[:2000],
        })

print(f"Loaded {len(techniques)} techniques, {len(groups)} threat groups")
spark.createDataFrame(techniques).write.mode("overwrite").saveAsTable("disa_workshop.threat_intel.attack_techniques")
spark.createDataFrame(groups).write.mode("overwrite").saveAsTable("disa_workshop.threat_intel.attack_groups")

## 5. Synthetic affected assets

Creates a fake DoD asset inventory of ~500 hosts so attendees can write join queries ("which KEV-listed CVEs affect our Windows servers?").

In [ ]:
import random
from datetime import date

random.seed(42)

VENDORS = ["Microsoft", "Cisco", "Adobe", "VMware", "Apache", "Fortinet", "Citrix"]
PRODUCTS = {
    "Microsoft": ["Windows Server 2019", "Exchange Server", "Office 365"],
    "Cisco": ["IOS XE", "ASA", "Catalyst Switch"],
    "Adobe": ["Acrobat Reader", "ColdFusion"],
    "VMware": ["vCenter", "ESXi"],
    "Apache": ["HTTP Server", "Log4j", "Struts"],
    "Fortinet": ["FortiOS", "FortiGate"],
    "Citrix": ["NetScaler ADC", "XenApp"],
}
ENVS = ["NIPRNet", "SIPRNet", "JWICS"]

assets = []
for i in range(500):
    vendor = random.choice(VENDORS)
    product = random.choice(PRODUCTS[vendor])
    assets.append({
        "asset_id": f"AST-{i:05d}",
        "hostname": f"host-{i:04d}.disa.mil",
        "environment": random.choice(ENVS),
        "vendor": vendor,
        "product": product,
        "os_family": "Windows" if vendor == "Microsoft" else "Linux" if vendor in ("Apache",) else "Network",
        "last_patched": str(date(2026, random.randint(1, 4), random.randint(1, 28))),
        "owner_org": random.choice(["DISA-J3", "DISA-J6", "USCYBERCOM", "JFHQ-DODIN"]),
    })

df = spark.createDataFrame(assets).withColumn("last_patched", F.to_date("last_patched"))
df.write.mode("overwrite").saveAsTable("disa_workshop.threat_intel.affected_assets")
print(f"Created {df.count()} synthetic assets")

## 6. Sample CISA advisory PDFs

We bundle ~20 real CISA advisory PDFs in `data/advisories/`. Upload them to the volume so Module 1 can call `ai_parse_document` on them.

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent  # notebooks/ -> repo root
ADVISORY_DIR = REPO_ROOT / "data" / "advisories"
VOLUME = "/Volumes/disa_workshop/threat_intel/raw_advisories"

if ADVISORY_DIR.exists():
    for pdf in ADVISORY_DIR.glob("*.pdf"):
        target = f"{VOLUME}/{pdf.name}"
        with open(pdf, "rb") as f:
            dbutils.fs.put(target, f.read().decode("latin-1"), overwrite=True)
    print(f"Uploaded PDFs from {ADVISORY_DIR} to {VOLUME}")
else:
    print(f"No PDFs at {ADVISORY_DIR}; skipping. Module 1 will download a few live samples instead.")

## 7. Verify

Quick row-count check across all seed tables.

In [ ]:
%sql
SELECT 'kev_catalog' AS tbl, COUNT(*) AS rows FROM disa_workshop.threat_intel.kev_catalog
UNION ALL SELECT 'cves', COUNT(*) FROM disa_workshop.threat_intel.cves
UNION ALL SELECT 'attack_techniques', COUNT(*) FROM disa_workshop.threat_intel.attack_techniques
UNION ALL SELECT 'attack_groups', COUNT(*) FROM disa_workshop.threat_intel.attack_groups
UNION ALL SELECT 'affected_assets', COUNT(*) FROM disa_workshop.threat_intel.affected_assets;

Setup complete. Proceed to **Module 1 — `notebooks/01_ingest_advisories.ipynb`**.